In [1]:
import logging
logging.basicConfig(level=logging.INFO)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("huggingface_hub").setLevel(logging.WARNING)

from rag.pipeline import RAGPipeline
pipeline = RAGPipeline()
print("chunks in index:", pipeline.store.count())

INFO:rag.embeddings:Loading embedding model: all-MiniLM-L6-v2
INFO:sentence_transformers.base.model:No device provided, using cpu
INFO:sentence_transformers.base.model:Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

INFO:rag.vector_store:Vector store ready: documents_cosine (322 chunks)
INFO:sentence_transformers.base.model:No device provided, using cpu
INFO:sentence_transformers.base.model:No modules.json found for cross-encoder/ms-marco-MiniLM-L-6-v2, initializing a new CrossEncoder model.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

chunks in index: 322


In [2]:
import json
for q in json.load(open("eval/questions.json")):
    if not q.get("answerable", True):
        hits = pipeline.retriever.retrieve(q["question"])
        if hits:
            print("\nPASSED RETRIEVAL:", q["question"])
            print("  top chunk: page", hits[0].page, "| score", round(hits[0].score, 2), "|", hits[0].source)
            print("  chunk text:", hits[0].text[:200].replace("\n", " "))
            print("  LLM says  :", pipeline.ask(q["question"]).text[:200])


PASSED RETRIEVAL: How do I deploy a RAG application with Kubernetes?
  top chunk: page 16 | score 0.68 | research2.pdf
  chunk text: RAG’s application scope is expanding into multimodal do- mains, adapting its principles to interpret and process diverse data forms like images, videos, and code. This expansion high- lights RAG’s sig
  LLM says  : I could not find this in the uploaded documents.

PASSED RETRIEVAL: What is the best GPU for training large language models?
  top chunk: page 10 | score 0.89 | research2.pdf
  chunk text: context can introduce more noise, diminishing the LLM’s perception of key information . (Long) LLMLingua [100], [101] utilize small language models (SLMs) such as GPT-2 Small or LLaMA-7B, to detect an
  LLM says  : I could not find this in the uploaded documents.
